# Router Chains — versão moderna (LCEL)

Substitui as classes deprecadas:
- `LLMChain` → `prompt | llm | StrOutputParser()`
- `MultiPromptChain` + `LLMRouterChain` → `RunnableLambda` + `RunnableBranch`
- `RouterOutputParser` → parsing manual de JSON
- `MULTI_PROMPT_ROUTER_TEMPLATE` → template próprio

In [29]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from dotenv import load_dotenv
import json

load_dotenv()

True

In [2]:
# Prompts especializados por matéria (sem alteração de conteúdo)
quimica_template = ChatPromptTemplate.from_template("""Você é um químico muito experiente.
Você é excelente em responder perguntas sobre química de forma clara e objetiva.
Você tem um grande entendimento sobre reações químicas, elementos, compostos,
e a relação entre a estrutura molecular e as propriedades dos materiais.
Quando você não sabe a resposta para uma pergunta, você admite que não sabe.

Aqui está uma pergunta: {input}""")

geografia_template = ChatPromptTemplate.from_template("""Você é um geógrafo muito bem informado.
Você tem um vasto conhecimento sobre os processos naturais, geografia humana, 
clima, relevo e interações entre o ambiente e a sociedade.
Você é habilidoso em explicar como fatores físicos e humanos afetam
o mundo ao nosso redor.

Aqui está uma pergunta: {input}""")

biologia_template = ChatPromptTemplate.from_template("""Você é um biólogo muito capacitado.
Você tem um grande conhecimento sobre os seres vivos, suas estruturas, funções
e a interação entre diferentes organismos e seus ambientes.
Você é excelente em explicar conceitos de biologia de maneira clara,
tanto para iniciantes quanto para estudantes avançados.

Aqui está uma pergunta: {input}""")

In [3]:
# Metadados de cada rota: nome, descrição e o template associado
prompt_infos = [
    {
        "name": "Química",
        "description": "Ideal para responder pergunta de química",
        "prompt_template": quimica_template
    },
    {
        "name": "Geografia",
        "description": "Ideal para responder pergunta de geografia",
        "prompt_template": geografia_template
    },
    {
        "name": "Biologia",
        "description": "Ideal para responder pergunta de biologia",
        "prompt_template": biologia_template
    },
]

In [9]:
chat = ChatOpenAI(model="gpt-3.5-turbo-0125")

In [10]:
# ✅ Chains de destino com LCEL (substitui LLMChain)
# Cada chain é: prompt | llm | parser de string
chains_destino = {}
for info in prompt_infos:
    chain = info["prompt_template"] | chat | StrOutputParser()
    chains_destino[info["name"]] = chain

chains_destino

{'Química': ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='Você é um químico muito experiente.\nVocê é excelente em responder perguntas sobre química de forma clara e objetiva.\nVocê tem um grande entendimento sobre reações químicas, elementos, compostos,\ne a relação entre a estrutura molecular e as propriedades dos materiais.\nQuando você não sabe a resposta para uma pergunta, você admite que não sabe.\n\nAqui está uma pergunta: {input}'), additional_kwargs={})])
 | ChatOpenAI(output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x00000201E73F98E0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000201E689DF40>, root_client=<openai.OpenAI object at 0x00000201E6898770>, root_async_client=<openai.AsyncOpenAI object at 0x00000201E73

In [11]:
# Lista de destinos formatada para o prompt do roteador
destinos = [f'{p["name"]}: {p["description"]}' for p in prompt_infos]
destinos_str = "\n".join(destinos)
print(destinos_str)

Química: Ideal para responder pergunta de química
Geografia: Ideal para responder pergunta de geografia
Biologia: Ideal para responder pergunta de biologia


In [17]:
# ✅ Substitua as células do router_template por isso:

router_template_str = (
    "Dado o input do usuário, selecione o prompt mais adequado entre os candidatos abaixo.\n"
    "Retorne APENAS um objeto JSON válido, sem markdown, sem texto extra.\n\n"
    'Formato obrigatório: {{"destination": "<nome>", "next_inputs": "<pergunta>"}}\n\n'
    f"<< CANDIDATOS >>\n{destinos_str}\n\n"
    'LEMBRE-SE: "destination" deve ser um dos nomes acima OU "DEFAULT".\n\n'
    "<< INPUT >>\n{input}\n"
)

router_prompt = PromptTemplate(
    template=router_template_str,
    input_variables=["input"]
)

print(router_prompt.template)

Dado o input do usuário, selecione o prompt mais adequado entre os candidatos abaixo.
Retorne APENAS um objeto JSON válido, sem markdown, sem texto extra.

Formato obrigatório: {{"destination": "<nome>", "next_inputs": "<pergunta>"}}

<< CANDIDATOS >>
Química: Ideal para responder pergunta de química
Geografia: Ideal para responder pergunta de geografia
Biologia: Ideal para responder pergunta de biologia

LEMBRE-SE: "destination" deve ser um dos nomes acima OU "DEFAULT".

<< INPUT >>
{input}



In [30]:
def parse_router_output(input_and_message: dict) -> dict:
    original_input = input_and_message["input"]      # salva o original
    ai_message     = input_and_message["ai_message"] # AIMessage do LLM

    text = ai_message.content.strip()
    text = text.replace("```json", "").replace("```", "").strip()
    parsed = json.loads(text)

    # Ignora o next_inputs do LLM — sempre usa o input original
    parsed["next_inputs"] = original_input
    return parsed

# Ajuste o router_chain para passar o input original junto com a resposta do LLM
router_chain = (
    RunnablePassthrough.assign(ai_message = router_prompt | chat)
    | RunnableLambda(parse_router_output)
)

In [31]:
# ✅ Chain padrão (fallback) para perguntas fora das matérias cadastradas
default_chain = ChatPromptTemplate.from_template("{input}") | chat | StrOutputParser()

# ✅ Função de despacho (substitui MultiPromptChain)
# Recebe o JSON do roteador e aciona a chain certa
def route(router_output: dict) -> str:
    destination = router_output.get("destination", "DEFAULT")
    next_input  = router_output.get("next_inputs", "")

    print(f"\n→ Roteando para: {destination!r}")
    print(f"→ Input enviado: {next_input!r}\n")

    if destination in chains_destino:
        return chains_destino[destination].invoke({"input": next_input})

    # Nenhuma rota encontrada → chain padrão
    return default_chain.invoke({"input": next_input})

# Pipeline completo: router_chain decide → route() despacha → chain especializada responde
chain = router_chain | RunnableLambda(route)

In [36]:
chain.invoke({"input": "O que é o El Niño?"})


→ Roteando para: 'Geografia'
→ Input enviado: 'O que é o El Niño?'



'O El Niño é um fenômeno climático natural que ocorre de forma cíclica no Oceano Pacífico. Ele se caracteriza pelo aquecimento anormal das águas superficiais do oceano na região equatorial, causando alterações significativas no clima global. Isso normalmente resulta em mudanças nos padrões de chuva, temperatura e ventos, afetando os ecossistemas marinhos e terrestres, bem como a agricultura, pesca e economia em diversas regiões do mundo. O El Niño geralmente acontece a cada 2 a 7 anos e pode durar de vários meses a um ano.'

In [21]:
chain.invoke({"input": "Pra que serve a tabela periódica?"})


→ Roteando para: 'Química'
→ Input enviado: 'Qual é a estrutura da tabela periódica?'



'A tabela periódica é organizada em linhas horizontais chamadas períodos e colunas verticais chamadas grupos. Cada elemento na tabela periódica é representado por um símbolo químico e possui propriedades únicas com base na sua estrutura atômica. Os elementos são organizados de acordo com seu número atômico, que é o número de prótons em seu núcleo. A estrutura da tabela periódica reflete a tendência dos elementos em termos de propriedades químicas e físicas, permitindo a previsão do comportamento dos elementos e facilitando a compreensão da química e suas interações.'

In [22]:
chain.invoke({"input": "Pra que serve a fotossíntese?"})


→ Roteando para: 'Biologia'
→ Input enviado: 'Qual a importância da fotossíntese para os seres vivos?'



'A fotossíntese é um processo fundamental para a vida na Terra, pois é a principal forma de produção de energia para a maioria dos seres vivos. Durante a fotossíntese, as plantas, algas e algumas bactérias utilizam a energia da luz solar para converter dióxido de carbono e água em glicose e oxigênio.\n\nA glicose produzida durante a fotossíntese é a principal fonte de energia para as plantas, que a utilizam para crescer, se reproduzir e realizar suas atividades metabólicas. Além disso, a glicose também é uma fonte de energia para os animais herbívoros que se alimentam de plantas.\n\nO oxigênio liberado durante a fotossíntese é essencial para a respiração aeróbica de todos os seres vivos, incluindo plantas, animais e muitos microorganismos. Durante a respiração, os organismos utilizam o oxigênio para oxidar a glicose e produzir energia.\n\nDessa forma, a fotossíntese desempenha um papel crucial na manutenção da vida na Terra, pois fornece energia para os seres vivos e também ajuda a man

In [23]:
# Teste de fallback — pergunta fora das matérias cadastradas
chain.invoke({"input": "Qual é a capital da Austrália?"})


→ Roteando para: 'Geografia'
→ Input enviado: 'Qual o continente da Austrália?'



'A Austrália é um país e um continente em si mesmo, localizado na Oceania. É o menor continente do mundo em termos de extensão territorial, mas também é considerado um continente devido à sua separação geográfica das demais massas de terra. Portanto, a resposta correta seria que a Austrália é o continente da Austrália.'

In [34]:
# Teste de fallback — pergunta fora das matérias cadastradas
chain.invoke({"input": "Qual é o sentido da vida?"})


→ Roteando para: 'DEFAULT'
→ Input enviado: 'Qual é o sentido da vida?'



'Essa é uma pergunta muito complexa e subjetiva, e a resposta pode variar de pessoa para pessoa. Alguns acreditam que o sentido da vida está ligado a encontrar a felicidade e o propósito, outros acreditam que é cumprir um papel ou missão específica, enquanto outros acreditam que o sentido da vida está relacionado à busca do conhecimento e evolução pessoal. Em geral, cada indivíduo deve refletir sobre suas próprias crenças, valores e objetivos para encontrar o seu próprio sentido da vida.'